In [0]:
# ===================================================
# BLOCK 1 — IMPORTS AND CONFIGURATION
# ===================================================

from pyspark.sql import Window
from pyspark.sql import functions as F


"""
Validate Gold dimension populations, surrogate keys, unknown members,
hierarchies, and SCD Type 2 effective-date behavior.
"""

GOLD = "semiconplus_portfolio.gold"

DIMENSIONS = {
    "date": (f"{GOLD}.dim_date", "date_key"),
    "site": (f"{GOLD}.dim_site", "site_key"),
    "product_group": (f"{GOLD}.dim_product_group", "product_group_key"),
    "equipment": (f"{GOLD}.dim_equipment", "equipment_key"),
    "defect": (f"{GOLD}.dim_defect", "defect_key"),
    "test_program": (f"{GOLD}.dim_test_program", "test_program_key"),
    "device": (f"{GOLD}.dim_device_scd2", "device_key"),
}

In [0]:
# ===================================================
# BLOCK 2 — TABLE, KEY, AND UNKNOWN-MEMBER VALIDATION
# ===================================================

"""
Confirm that each dimension exists, has unique surrogate keys, and
contains exactly one unknown member with surrogate key 0.
"""

for dimension_name, (table_name, key_column) in DIMENSIONS.items():
    assert spark.catalog.tableExists(table_name)

    dimension_df = spark.table(table_name)

    duplicate_key_count = (
        dimension_df
        .groupBy(key_column)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    unknown_member_count = dimension_df.filter(
        F.col(key_column) == 0
    ).count()

    print(
        f"{dimension_name}: rows={dimension_df.count():,}, "
        f"duplicate keys={duplicate_key_count}, "
        f"unknown members={unknown_member_count}"
    )

    assert duplicate_key_count == 0
    assert unknown_member_count == 1

print("Dimension surrogate-key and unknown-member validation passed.")

In [0]:
# ===================================================
# BLOCK 3 — SNAPSHOT DIMENSION POPULATIONS
# ===================================================

"""
Confirm that snapshot dimensions contain all validated source members plus
their documented unknown member.
"""

assert spark.table(f"{GOLD}.dim_site").count() == 4
assert spark.table(f"{GOLD}.dim_product_group").count() == 7
assert spark.table(f"{GOLD}.dim_equipment").count() == 25

assert spark.table(f"{GOLD}.dim_date").filter(
    F.col("calendar_date").between("2021-01-01", "2026-12-31")
).count() == 2_191

print("Snapshot dimension population validation passed.")


In [0]:
# ===================================================
# BLOCK 4 — DIMENSION HIERARCHY VALIDATION
# ===================================================

"""
Confirm that every known equipment member resolves to a known Gold site
member and retains a consistent business site identifier.
"""

equipment_df = spark.table(f"{GOLD}.dim_equipment").filter(
    F.col("equipment_key") != 0
)
site_df = spark.table(f"{GOLD}.dim_site").filter(F.col("site_key") != 0)

equipment_site_failures = (
    equipment_df.alias("e")
    .join(site_df.alias("s"), "site_key", "left")
    .filter(
        F.col("s.site_key").isNull()
        | (F.col("e.site_id") != F.col("s.site_id"))
    )
    .count()
)

assert equipment_site_failures == 0

print("Gold dimension hierarchy validation passed.")

In [0]:
# ===================================================
# BLOCK 5 — DEVICE SCD CURRENT-ROW VALIDATION
# ===================================================

"""
Confirm that every known device business key has exactly one current
version and that the controlled changed device has two historical rows.
"""

device_df = spark.table(f"{GOLD}.dim_device_scd2")
known_devices_df = device_df.filter(F.col("device_key") != 0)

invalid_current_counts = (
    known_devices_df
    .groupBy("device_id")
    .agg(F.sum(F.col("is_current").cast("int")).alias("current_count"))
    .filter(F.col("current_count") != 1)
    .count()
)

assert invalid_current_counts == 0
assert known_devices_df.select("device_id").distinct().count() == 30
assert known_devices_df.filter(F.col("device_id") == "DV001").count() == 2
assert known_devices_df.filter(F.col("device_id") != "DV001").count() == 29

print("Device SCD current-row validation passed.")

In [0]:
# ===================================================
# BLOCK 6 — SCD DATE-RANGE VALIDATION
# ===================================================

"""
Confirm that device effective-date ranges are valid, contiguous for the
controlled change, and do not overlap within a business key.
"""

scd_window = Window.partitionBy("device_id").orderBy("effective_from")

scd_ranges_df = (
    known_devices_df
    .withColumn("next_effective_from", F.lead("effective_from").over(scd_window))
)

invalid_ranges = scd_ranges_df.filter(
    F.col("effective_from") > F.col("effective_to")
).count()

overlapping_ranges = scd_ranges_df.filter(
    F.col("next_effective_from").isNotNull()
    & (F.col("effective_to") >= F.col("next_effective_from"))
).count()

assert invalid_ranges == 0
assert overlapping_ranges == 0

dv001_history = (
    known_devices_df
    .filter(F.col("device_id") == "DV001")
    .orderBy("effective_from")
    .collect()
)

assert str(dv001_history[0]["effective_from"]) == "2021-01-01"
assert str(dv001_history[0]["effective_to"]) == "2025-12-31"
assert dv001_history[0]["is_current"] is False
assert str(dv001_history[1]["effective_from"]) == "2026-01-01"
assert str(dv001_history[1]["effective_to"]) == "9999-12-31"
assert dv001_history[1]["is_current"] is True
assert dv001_history[1]["version_number"] == 2

display(
    known_devices_df
    .filter(F.col("device_id") == "DV001")
    .orderBy("effective_from")
)

print("Device SCD effective-date validation passed.")

In [0]:
# ===================================================
# BLOCK 7 — DEFECT AND PROGRAM DIMENSION VALIDATION
# ===================================================

"""
Confirm that every accepted test-result defect and program combination
resolves to a Gold dimension member.
"""

tests_df = spark.table("semiconplus_portfolio.silver.unit_test_results")
defect_df = spark.table(f"{GOLD}.dim_defect")
program_df = spark.table(f"{GOLD}.dim_test_program")

unresolved_defects = (
    tests_df.select("defect_code").distinct()
    .join(defect_df.select("defect_code"), "defect_code", "left_anti")
    .count()
)

unresolved_programs = (
    tests_df.select("device_id", "program_revision").distinct()
    .join(
        program_df.select("device_id", "program_revision"),
        ["device_id", "program_revision"],
        "left_anti",
    )
    .count()
)

assert unresolved_defects == 0
assert unresolved_programs == 0

print("Defect and test-program dimension validation passed.")

In [0]:
# ===================================================
# BLOCK 8 — CAPTURE RERUN BASELINE
# ===================================================

"""
Capture dimension populations before repeating the Gold build.
"""

counts_before_rerun = {
    name: spark.table(table_name).count()
    for name, (table_name, _) in DIMENSIONS.items()
}

print(counts_before_rerun)

In [0]:
# ---------------
# RERUN PROCEDURE
# ---------------

# 1. Run Block 8.
# 2. Rerun all blocks in 11_gold_dimensions.
# 3. Return here without clearing this notebook session.
# 4. Run Block 9.

# ===================================================
# BLOCK 9 — IDEMPOTENCY VALIDATION
# ===================================================

"""
Confirm that snapshot dimensions remain stable and the controlled SCD
version is not inserted again during a rerun.
"""

counts_after_rerun = {
    name: spark.table(table_name).count()
    for name, (table_name, _) in DIMENSIONS.items()
}

assert counts_after_rerun == counts_before_rerun

assert spark.table(f"{GOLD}.dim_device_scd2").filter(
    F.col("device_id") == "DV001"
).count() == 2

print("GOLD DIMENSION IDEMPOTENCY TEST PASSED")

In [0]:
# ===================================================
# BLOCK 10 — FINAL VALIDATION RESULT
# ===================================================

"""
Publish the final Gold dimension validation result for project evidence.
"""

print("GOLD DIMENSION VALIDATION PASSED")
print("Known sites: 3")
print("Known product groups: 6")
print("Known equipment: 24")
print("Known devices: 30")
print("DV001 historical versions: 2")
print("SCD overlapping ranges: 0")
print("Unresolved defects: 0")
print("Unresolved test programs: 0")
